# 02 · Tool, filesystem ed esecuzione

Gli agenti diventano utili quando **agiscono** sul mondo: leggono e scrivono file, eseguono
codice. Qui vediamo tre cose, con attenzione alla **sicurezza**:
1. un tool tipizzato ben definito;
2. tool su file **confinati** in una cartella (niente path traversal);
3. eseguire codice in un **sottoprocesso** isolato con timeout.

## Obiettivi, prerequisiti e modalità di lettura

Costruirai tool testabili, un workspace confinato e un esecutore con timeout. Durata: 30–40 minuti. Il sottoprocesso è didattico e non sostituisce Docker contro codice ostile.

Ogni blocco di codice è preceduto da una spiegazione e seguito da un **output
atteso**. Quando interviene un modello, l'output atteso descrive proprietà e
invarianti, non una frase letterale. Esegui le celle in ordine e non saltare i
casi negativi: mostrano il confine del meccanismo, non un incidente del corso.

## Setup (autonomo)

Ogni notebook è **indipendente**: non importa nulla dal progetto. Qui carichiamo la chiave
API dal file `.env` e creiamo un modello. Esegui le celle in ordine dall'alto verso il basso.

### Spiegazione del blocco · Caricamento della configurazione

Il setup è locale al notebook e non dipende da moduli del repository. La ricerca ascendente rende stabile il percorso di `.env`.

In [ ]:
# Carichiamo le variabili d'ambiente dal file `.env`.
# Lo cerchiamo nella cartella corrente e in quelle superiori, così il notebook
# funziona sia se avviato dalla radice del progetto sia dalla cartella `notebooks`.
import os
from pathlib import Path

from dotenv import load_dotenv


def trova_env() -> Path:
    for cartella in (Path.cwd(), *Path.cwd().resolve().parents):
        if (cartella / ".env").is_file():
            return cartella / ".env"
    raise FileNotFoundError("File .env non trovato: copia .env.example in .env e aggiungi la chiave.")


env_file = trova_env()
load_dotenv(env_file, override=False)          # carica le variabili senza sovrascrivere quelle già presenti
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY mancante nel file .env"
print("Ambiente caricato da:", env_file)

### Output atteso

Percorso del file `.env`, oppure errore immediato se configurazione assente.

### Spiegazione del blocco · Inizializzazione del modello

Viene creato un solo client e riusato nelle sezioni successive, evitando configurazioni divergenti tra esempi.

In [ ]:
# `ChatOpenAI` è il wrapper LangChain attorno al modello.
# Lo creiamo una volta e lo riusiamo in tutto il notebook.
from langchain_openai import ChatOpenAI

MODELLO = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")   # modello economico, va bene per imparare
model = ChatOpenAI(
    model=MODELLO,
    use_responses_api=True,   # API "responses" di OpenAI
    store=False,              # non conservare la conversazione sui server OpenAI
)
print("Modello pronto:", MODELLO)

### Output atteso

`Modello pronto: <nome-modello>`.

## 1 · Un tool tipizzato

Il modello sceglie un tool leggendo nome, tipi e docstring. Più sono chiari, meglio sceglie.

### Spiegazione del blocco · Tool puro e test diretto

Prima di coinvolgere un agente si verifica il tool isolatamente. Questo separa errori della funzione da errori di scelta del modello.

In [ ]:
from langchain_core.tools import tool


@tool
def conta_parole(testo: str) -> int:
    """Conta quante parole ci sono in un testo."""
    return len(testo.split())


# Possiamo provare un tool anche senza agente, con `.invoke`.
print(conta_parole.invoke({"testo": "uno due tre"}))

### Output atteso

Il numero `3`, perché la frase contiene tre parole.

## 2 · File confinati in una cartella (workspace)

Un agente che scrive file è potente ma pericoloso. La difesa: consentire **solo** una
cartella "workspace" e rifiutare qualsiasi percorso che ne esca (es. `../../etc/passwd`).

### Spiegazione del blocco · Creazione del workspace confinato

Il path viene normalizzato con `resolve()` e poi confrontato con la root. Controllare la stringa prima della normalizzazione non basterebbe contro sequenze `..`.

In [ ]:
#import tempfile
from datetime import datetime
from pathlib import Path

# WORKSPACE IN CARTELLA TEMPORANEA
# Creiamo una cartella temporanea che fa da "workspace" sicuro.
# WORKSPACE = Path(tempfile.mkdtemp(prefix="nb_workspace_"))

# WORKSPACE IN CARTELLA NOTEBOOKS
def trova_cartella_notebook() -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == "notebooks":
        return cwd
    candidato = cwd / "notebooks"
    if candidato.is_dir():
        return candidato
    return cwd

# Sottocartella unica dentro notebooks/workspace con prefisso nb_workspace_
WORKSPACE_ROOT = (trova_cartella_notebook() / "workspace").resolve()
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d%H%M%S%f")
WORKSPACE = (WORKSPACE_ROOT / f"nb_workspace_{timestamp}").resolve()
WORKSPACE.mkdir(parents=True, exist_ok=True)
print("Workspace:", WORKSPACE)
# FINE WORKSPACE IN CARTELLA NOTEBOOKS

def percorso_sicuro(nome: str) -> Path:
    # `resolve()` normalizza il path (risolve i `..`); poi controlliamo che resti DENTRO il workspace.
    candidato = (WORKSPACE / nome).resolve()
    if not candidato.is_relative_to(WORKSPACE.resolve()):
        raise ValueError(f"Percorso fuori dal workspace: {nome}")
    return candidato


### Output atteso

Un percorso temporaneo con prefisso `nb_workspace_`. Nessun file viene ancora creato.

### Spiegazione del blocco · Test negativo del path traversal

Il caso malevolo viene provato intenzionalmente. Una difesa è utile solo se il percorso vietato produce un errore osservabile.

In [ ]:
# Verifica veloce della difesa: un percorso malevolo viene bloccato.
try:
    percorso_sicuro("../fuori.txt")
except ValueError as errore:
    print("Bloccato correttamente:", errore)

### Output atteso

`Bloccato correttamente: Percorso fuori dal workspace: ../fuori.txt`.

### Spiegazione del blocco · Tool di lettura e scrittura

Entrambi passano dalla stessa funzione di confinement. Centralizzare la guardia evita che un tool dimentichi il controllo.

In [ ]:
# Ora i due tool su file: entrambi passano da `percorso_sicuro`.
# Gli errori tornano come stringa: così l'agente li vede come osservazione, non crasha la cella.
@tool
def scrivi_file(nome: str, contenuto: str) -> str:
    """Scrive un file di testo nel workspace. Usa solo nomi relativi (es. somma.py)."""
    try:
        percorso = percorso_sicuro(nome)
        percorso.parent.mkdir(parents=True, exist_ok=True)
        percorso.write_text(contenuto, encoding="utf-8")
        return f"Scritti {len(contenuto)} caratteri in {percorso.name} (workspace)"
    except ValueError as errore:
        return f"ERRORE: {errore}"


@tool
def leggi_file(nome: str) -> str:
    """Legge un file di testo dal workspace. Usa solo nomi relativi (es. somma.py)."""
    try:
        return percorso_sicuro(nome).read_text(encoding="utf-8")
    except (ValueError, FileNotFoundError) as errore:
        return f"ERRORE: {errore}"


### Output atteso

Nessun output. Sono disponibili `scrivi_file` e `leggi_file`.

## 3 · Eseguire codice in un sottoprocesso isolato

Per far "provare" del codice all'agente lo eseguiamo in un **processo separato**, con un
**timeout**. In produzione si usa un container (es. Docker); qui basta il concetto: l'output
(stdout, stderr, exit code) torna come *osservazione*, non blocca mai il programma.

### Spiegazione del blocco · Esecuzione con timeout

Il sottoprocesso riceve argomenti come lista e non usa shell. Exit code, stdout e stderr restano distinti, così l'agente può verificare davvero il risultato.

In [ ]:
import subprocess


def _formatta_esito(risultato: subprocess.CompletedProcess[str]) -> str:
    # Exit code, stdout e stderr restano distinti: l'agente verifica davvero il risultato.
    return (
        f"exit={risultato.returncode}\n"
        f"STDOUT:\n{risultato.stdout}\n"
        f"STDERR:\n{risultato.stderr}"
    )


@tool
def esegui_python(codice: str) -> str:
    """Esegue un frammento Python inline (`python -c`) e restituisce l'output."""
    try:
        risultato = subprocess.run(
            ["python", "-c", codice],   # niente shell: passiamo gli argomenti come lista
            capture_output=True, text=True, timeout=10, check=False,
        )
    except subprocess.TimeoutExpired:
        return "ERRORE: esecuzione troppo lunga (timeout)."
    return _formatta_esito(risultato)


@tool
def esegui_file(nome: str) -> str:
    """Esegue un file .py già scritto nel workspace (stesso nome usato con scrivi_file)."""
    try:
        percorso = percorso_sicuro(nome)
        risultato = subprocess.run(
            ["python", str(percorso)],
            cwd=str(WORKSPACE),          # cwd = workspace: niente path assoluti inventati
            capture_output=True, text=True, timeout=10, check=False,
        )
    except ValueError as errore:
        return f"ERRORE: {errore}"
    except subprocess.TimeoutExpired:
        return "ERRORE: esecuzione troppo lunga (timeout)."
    return _formatta_esito(risultato)


### Output atteso

Nessun output. `esegui_python` è definito; un timeout restituirà un messaggio controllato.

## 4 · Un agente che usa i tool

Diamo tutti i tool all'agente e chiediamo un compito che richiede di scrivere, eseguire e verificare.

### Spiegazione del blocco · Assemblaggio dell'agente operativo

Il modello vede quattro strumenti con responsabilità diverse. Il prompt richiede una verifica dopo la scrittura del codice.

In [ ]:
from langchain.agents import create_agent

agente = create_agent(
    model=model,
    tools=[conta_parole, scrivi_file, leggi_file, esegui_python, esegui_file],
    system_prompt=(
        "I file vivono solo nel workspace temporaneo. "
        "Usa SEMPRE nomi relativi come somma.py (mai path assoluti o cartelle tipo notebooks/). "
        "Flusso obbligatorio: scrivi_file → esegui_file sullo stesso nome → "
        "controlla exit=0 e STDOUT prima di concludere."
    ),
)


### Output atteso

Nessun output. L'agente è pronto a leggere, scrivere ed eseguire Python.

### Spiegazione del blocco · Flusso write–execute–verify

La richiesta non può essere soddisfatta correttamente con solo testo: deve produrre un file, eseguirlo e interpretare l'exit code.

In [ ]:
esito = agente.invoke({"messages": [{
    "role": "user",
    "content": "Scrivi similarity.py che calcola la similarità tra due vettori da 10 elementi creati da te",
}]})
print(esito["messages"][-1].text)

for messaggio in esito["messages"]:
    print(messaggio.text)


### Output atteso

Una risposta che conferma risultato `55`. Nel workspace deve comparire `somma.py`; trace e formulazione dipendono dal modello.

## Prova tu

- Chiedi all'agente di leggere un file inesistente: vedrai l'errore tornare come osservazione.
- Aggiungi un tool `elenca_file` che lista il workspace.

**Idea chiave**: un tool è un confine di fiducia. Validare i percorsi e isolare l'esecuzione
è ciò che rende sicuro dare "mani" a un agente.

## Laboratorio aggiuntivo

Gli esempi seguenti riusano quanto costruito sopra. Il primo amplia il caso normale; il
secondo esercita un confine, un errore o una proprietà che spesso causa bug reali.

## Esempio aggiuntivo: tabella di path consentiti e vietati

### Spiegazione del blocco

Provare più forme evita una difesa costruita su un solo caso noto.

In [ ]:
for nome in ("report.txt", "cartella/dati.csv", "../segreto.txt", "/tmp/fuori.txt"):
    try:
        print(nome, "-> OK:", percorso_sicuro(nome))
    except ValueError as errore:
        print(nome, "-> BLOCCATO:", errore)

### Output atteso

I primi due path restano nel workspace; `../segreto.txt` e `/tmp/fuori.txt` vengono bloccati.

## Esempio aggiuntivo: distinguere successo ed errore del processo

### Spiegazione del blocco

Exit code e stderr sono segnali più affidabili di una frase del modello.

In [ ]:
print(esegui_python.invoke({"codice": "print(sum(range(6)))"}))
print(esegui_python.invoke({"codice": "raise RuntimeError('demo')"}))

### Output atteso

Prima esecuzione: `exit=0` e stdout `15`. Seconda: exit diverso da zero e traceback in stderr.

## Riepilogo e troubleshooting

Prima di proseguire, prova a spiegare con parole tue: quale stato è cambiato, quale
componente ha preso la decisione e quale prova rende osservabile l'esito.

Se una cella fallisce:

1. rileggi l'output atteso e individua la prima invariante non rispettata;
2. verifica di aver eseguito tutte le celle precedenti nello stesso kernel;
3. per i notebook live, controlla `.env`, modello disponibile e quota API;
4. riavvia il kernel solo dopo aver conservato eventuali file che vuoi ispezionare;
5. non correggere un caso negativo: l'errore previsto è parte dell'esempio.